In [1]:
# Data Preprocessing
# Import Libraries and Data Load
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Shape:", df.shape)
df.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
# info,  Missing values  and Duplicates check
df.info()
print("Duplicates:", df.duplicated().sum())
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [3]:
#  Missing values handle
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Missing before:")
print(df.isnull().sum()[df.isnull().sum() > 0])
df["TotalCharges"] = df["TotalCharges"].fillna(0)
print("\nMissing after:", df.isnull().sum().sum())

Missing before:
TotalCharges    11
dtype: int64

Missing after: 0


In [4]:
# Remove Duplicates
print("Duplicates before:", df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicates after:", df.duplicated().sum())
df = df.drop("customerID", axis=1)
print("Shape now:", df.shape)

Duplicates before: 0
Duplicates after: 0
Shape now: (7043, 20)


In [5]:
# Encoding
if df["Churn"].dtype == "object":
    df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})
cat_cols = df.select_dtypes(include="object").columns.tolist()
print("Categorical columns:", cat_cols)
df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

print("Shape now:", df.shape)
df.head()

Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Shape now: (7043, 31)


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,0,1,0,0,1,...,0,0,0,0,0,0,1,0,1,0
1,0,34,56.95,1889.50,0,1,0,0,1,0,...,0,0,0,0,1,0,0,0,0,1
2,0,2,53.85,108.15,1,1,0,0,1,0,...,0,0,0,0,0,0,1,0,0,1
3,0,45,42.30,1840.75,0,1,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
4,0,2,70.70,151.65,1,0,0,0,1,0,...,0,0,0,0,0,0,1,0,1,0


In [6]:
# Train/Test split and Scaling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train = X_train.copy()
X_test = X_test.copy()

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])
results = []

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print(X_train[num_cols].describe().round(2))

X_train: (5634, 30) | X_test: (1409, 30)
        tenure  MonthlyCharges  TotalCharges
count  5634.00         5634.00       5634.00
mean     -0.00           -0.00          0.00
std       1.00            1.00          1.00
min      -1.32           -1.54         -1.01
25%      -0.96           -0.97         -0.83
50%      -0.14            0.18         -0.40
75%       0.92            0.83          0.67
max       1.61            1.79          2.80


In [7]:
#  Model training and evaluation
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def evaluate_model(name, model):
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start

    y_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Training Time (s)": round(train_time, 3),
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
    })

    print(name)
    print(results[-1])
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

# Model 1: Logistic Regression
evaluate_model("Logistic Regression", LogisticRegression(max_iter=1000, random_state=42))

Logistic Regression
{'Model': 'Logistic Regression', 'Training Time (s)': 0.053, 'Accuracy': 0.8062, 'Precision': 0.6593, 'Recall': 0.5588, 'F1 Score': 0.6049}
Confusion Matrix:
[[927 108]
 [165 209]]


In [8]:
# Model 2: Decision Tree
from sklearn.tree import DecisionTreeClassifier

evaluate_model("Decision Tree", DecisionTreeClassifier(random_state=42))

Decision Tree
{'Model': 'Decision Tree', 'Training Time (s)': 0.122, 'Accuracy': 0.7253, 'Precision': 0.4825, 'Recall': 0.4786, 'F1 Score': 0.4805}
Confusion Matrix:
[[843 192]
 [195 179]]


In [9]:
# Model 3: Random Forest (100 trees)
from sklearn.ensemble import RandomForestClassifier

evaluate_model("Random Forest", RandomForestClassifier(n_estimators=100, random_state=42))

Random Forest
{'Model': 'Random Forest', 'Training Time (s)': 0.973, 'Accuracy': 0.785, 'Precision': 0.6187, 'Recall': 0.4947, 'F1 Score': 0.5498}
Confusion Matrix:
[[921 114]
 [189 185]]


In [10]:
# Model 4: KNN
from sklearn.neighbors import KNeighborsClassifier

evaluate_model("KNN", KNeighborsClassifier(n_neighbors=5))

KNN
{'Model': 'KNN', 'Training Time (s)': 0.006, 'Accuracy': 0.763, 'Precision': 0.5538, 'Recall': 0.5508, 'F1 Score': 0.5523}
Confusion Matrix:
[[869 166]
 [168 206]]


In [11]:
# Model 5: SVM
from sklearn.svm import SVC

evaluate_model("SVM", SVC(kernel="rbf", random_state=42))

SVM
{'Model': 'SVM', 'Training Time (s)': 1.29, 'Accuracy': 0.7949, 'Precision': 0.6545, 'Recall': 0.4813, 'F1 Score': 0.5547}
Confusion Matrix:
[[940  95]
 [194 180]]


In [12]:
# Model Comparison
comparison_df = pd.DataFrame(results)

# Sort According to score
comparison_df = comparison_df.sort_values(by="F1 Score", ascending=False).reset_index(drop=True)
comparison_df

,Model,Training Time (s),Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.053,0.8062,0.6593,0.5588,0.6049
1,SVM,1.290,0.7949,0.6545,0.4813,0.5547
2,KNN,0.006,0.7630,0.5538,0.5508,0.5523
3,Random Forest,0.973,0.7850,0.6187,0.4947,0.5498
4,Decision Tree,0.122,0.7253,0.4825,0.4786,0.4805


In [16]:
# Comparison Table (With Advantages and Limitations)
advantages = {
    "Logistic Regression": "Simple, tez, samajhne mein asaan; binary classification ke liye acha baseline",
    "Random Forest (Tuned)": "Tuning se default Random Forest se behtar; overfitting kam; Precision acha",
    "SVM": "Complex patterns pakar sakta hai; high-dimensional data par acha chalta hai",
    "KNN": "Samajhna asaan; training bohat tez (sirf data yaad rakhta hai)",
    "Random Forest": "Bohat saare trees ka vote, is liye ek tree se zyada stable; overfitting kam",
    "Decision Tree": "Rules ki shakal mein samajh aata hai; scaling ki zaroorat nahi",
}
limitations = {
    "Logistic Regression": "Sirf seedhe (linear) relationships pakar sakta hai",
    "Random Forest (Tuned)": "Tuning mein extra time lagta hai; Recall Logistic Regression se kam; samajhna mushkil",
    "SVM": "Training sust; bare dataset par mehenga; parameters tune karne parte hain",
    "KNN": "Prediction sust (har baar saare data se distance); noise se asar leta hai",
    "Random Forest": "Zyada memory aur time; samajhna mushkil",
    "Decision Tree": "Training data ratta laga leta hai (overfitting), is liye test par kamzor",
}

final_df = pd.DataFrame({
    "Model Name": comparison_df["Model"],
    "Training Time (s)": comparison_df["Training Time (s)"],
    "Evaluation Metrics": comparison_df.apply(
        lambda r: f"Acc: {r['Accuracy']} | Prec: {r['Precision']} | Rec: {r['Recall']} | F1: {r['F1 Score']}",
        axis=1),
    "Advantages": comparison_df["Model"].map(advantages),
    "Limitations": comparison_df["Model"].map(limitations),
})

final_df.style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"})

,Model Name,Training Time (s),Evaluation Metrics,Advantages,Limitations
0,Logistic Regression,0.053000,Acc: 0.8062 | Prec: 0.6593 | Rec: 0.5588 | F1: 0.6049,"Simple, tez, samajhne mein asaan; binary classification ke liye acha baseline",Sirf seedhe (linear) relationships pakar sakta hai
1,Random Forest (Tuned),0.503000,Acc: 0.8062 | Prec: 0.676 | Rec: 0.5187 | F1: 0.587,Tuning se default Random Forest se behtar; overfitting kam; Precision acha,Tuning mein extra time lagta hai; Recall Logistic Regression se kam; samajhna mushkil
2,SVM,1.290000,Acc: 0.7949 | Prec: 0.6545 | Rec: 0.4813 | F1: 0.5547,Complex patterns pakar sakta hai; high-dimensional data par acha chalta hai,Training sust; bare dataset par mehenga; parameters tune karne parte hain
3,KNN,0.006000,Acc: 0.763 | Prec: 0.5538 | Rec: 0.5508 | F1: 0.5523,Samajhna asaan; training bohat tez (sirf data yaad rakhta hai),Prediction sust (har baar saare data se distance); noise se asar leta hai
4,Random Forest,0.973000,Acc: 0.785 | Prec: 0.6187 | Rec: 0.4947 | F1: 0.5498,"Bohat saare trees ka vote, is liye ek tree se zyada stable; overfitting kam",Zyada memory aur time; samajhna mushkil
5,Decision Tree,0.122000,Acc: 0.7253 | Prec: 0.4825 | Rec: 0.4786 | F1: 0.4805,Rules ki shakal mein samajh aata hai; scaling ki zaroorat nahi,"Training data ratta laga leta hai (overfitting), is liye test par kamzor"


##  Conclusion

**Best model: Logistic Regression**

Logistic Regression performed best overall on the Telco Customer Churn dataset, with the highest F1 Score (0.6049), the highest Recall (0.5588) and the joint-highest Accuracy (0.8062). It was also the fastest of the strong models to train (0.053 seconds).

**Why it performed better:**
- Churn is largely explained by simple factors such as contract type, tenure and monthly charges, which a linear model captures well.
- Simple models are less likely to overfit. The Decision Tree, trained without any depth limit, memorised the training data and scored lowest on test data (F1: 0.4805).
- Random Forest (default) and SVM had lower Recall (0.4947 and 0.4813), and SVM was also much slower (1.29 seconds).

**Effect of hyperparameter tuning:**
GridSearchCV improved Random Forest from an F1 of 0.5498 to 0.5870 (best parameters: max_depth=10, min_samples_leaf=5, n_estimators=100). The tuned model matched Logistic Regression on Accuracy (0.8062) and had slightly higher Precision (0.6760), but its Recall (0.5187) and F1 Score were still lower. So tuning helped, but did not change the best model.

**Important note:** Even the best model has a Recall of only 0.5588, so it still misses about 44% of customers who actually churn (165 out of 374 in the test set). The data is also imbalanced (about 27% churn), which makes churners harder to detect.

In [14]:
# Bonus: Hyperparameter Tuning (GridSearchCV) - Random Forest
from sklearn.model_selection import GridSearchCV

# Settings ke combinations jo try karne hain
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_leaf": [1, 5],
}

# 5-fold cross-validation F1 Score ke hisaab se best chunna
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV F1 Score:", round(grid.best_score_, 4))

# Best settings wale model ko test data par evaluate karna
evaluate_model("Random Forest (Tuned)", grid.best_estimator_)

Best parameters: {'max_depth': 10, 'min_samples_leaf': 5, 'n_estimators': 100}
Best CV F1 Score: 0.5682
Random Forest (Tuned)
{'Model': 'Random Forest (Tuned)', 'Training Time (s)': 0.503, 'Accuracy': 0.8062, 'Precision': 0.676, 'Recall': 0.5187, 'F1 Score': 0.587}
Confusion Matrix:
[[942  93]
 [180 194]]


In [15]:
# Tuning ke baad: comparison table update (ab 6 models hain)
comparison_df = pd.DataFrame(results).sort_values(by="F1 Score", ascending=False).reset_index(drop=True)
comparison_df

,Model,Training Time (s),Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.053,0.8062,0.6593,0.5588,0.6049
1,Random Forest (Tuned),0.503,0.8062,0.6760,0.5187,0.5870
2,SVM,1.290,0.7949,0.6545,0.4813,0.5547
3,KNN,0.006,0.7630,0.5538,0.5508,0.5523
4,Random Forest,0.973,0.7850,0.6187,0.4947,0.5498
5,Decision Tree,0.122,0.7253,0.4825,0.4786,0.4805
